# Task 1 — Article Type Classification

Which method can recognise a fashion product's article type from its image?

We compared a small CNN trained from scratch with two models using hand-built image features. We then tested image augmentation and class weighting, studied the errors, and chose a model using development results. This notebook explains the questions behind those tests and what we learned.

> **Running this notebook:** The saved outputs show the completed experiments discussed here. The default Run All performs short smoke checks, so it does not reproduce the full experiment tables. Full training must be selected explicitly. Run records and settings are in `results/runs.csv`; supporting tables are in `results/evidence/task1/`.

**File execution flow**

| Order | Producer or implementation | Artifact passed forward | Consumer and effect |
|---:|---|---|---|
| 1 | [Canonical split](../data/processed/splits.csv), [label map](../data/processed/label_maps.json), and [shared loader](../src/fashion/data/dataset.py) | Fixed development rows, five folds, image paths, and 124-class order | [Task 1 dataset and preprocessing code](../src/fashion/task1/) builds fold-safe inputs; changing either data contract invalidates every comparison. |
| 2 | [Candidate declarations](../src/fashion/task1/candidates.py) and the CNN, weighted-loss, and classical experiment runners in [src/fashion/task1](../src/fashion/task1/) | One explicit candidate recipe and run mode | The current notebook starts smoke checks by default. Deliberate CNN runs use `mode="full"`; the classical branch uses `stage="tune"` and then `stage="final"`, without changing the shared folds. |
| 3 | [CNN engine](../src/fashion/task1/cnn_engine.py), [CNN training](../src/fashion/task1/training.py), and [classical training](../src/fashion/task1/classical_training.py) | Fold checkpoints, histories, predictions, and metrics | [Task 1 registry adapter](../src/fashion/task1/registry.py) records real runs in [results/runs.csv](../results/runs.csv), while compact evidence is written under [results/evidence/task1](../results/evidence/task1/). |
| 4 | Saved run and evidence artifacts | Five-fold, pooled OOF, learning-curve, weak-class, and confusion evidence | This notebook compares the candidates and freezes `task1_cnn_no_aug_unweighted_v1`; evidence describes the trained models but never changes them. |
| 5 | Frozen development choice | Selected recipe and provenance | [Task 1 final evaluation](03_task1_part2_final_evaluation.ipynb) calls [refit.py](../src/fashion/task1/refit.py) and [final_evaluation_runner.py](../src/fashion/task1/final_evaluation_runner.py) to verify the final bundle, evaluate it, and create the `articleType` test output. |

**End-to-end direction:** canonical data → declared candidate → fold training → registry and evidence → development choice in this notebook → final refit, evaluation, and export in Notebook 03. Final-evaluation results do not flow back into model selection.

## 1. Problem and output

The input is one fashion product image. The output is one of 124 `articleType` labels, such as `Tshirts`, `Tops`, or `Sports Shoes`. This prediction fills the `articleType` column in the assignment's `id,gender,articleType,season,usage` file.

The model must learn from the supplied images. All CNN candidates started with random weights; no pretrained weights were used. The main challenge is to recognise both common products and rare article types from small images.

## 2. EDA evidence

We first inspected the development data to find problems that could affect learning. The tables below count products per class, image formats, and classes with too few examples for the saved folds. Each finding gives a reason for a model or preprocessing choice to test.

In [1]:
from dataclasses import asdict

import pandas as pd

from fashion.config import DEVELOPMENT_CLASS_SUMMARY_CSV
from fashion.data.dataset import get_samples, load_label_maps, load_splits
from fashion.task1 import (
    build_task1_decision_evidence,
    build_task1_problem_profile,
)

TARGET = "articleType"
splits = load_splits()
task1_development = get_samples(splits, partition="development", target=TARGET)
article_type_map = load_label_maps()[TARGET]
article_type_classes = tuple(article_type_map["classes"])
class_summary = pd.read_csv(DEVELOPMENT_CLASS_SUMMARY_CSV, keep_default_na=False)
problem_profile = build_task1_problem_profile(splits, class_summary)

display(pd.DataFrame([asdict(problem_profile)]))
display(build_task1_decision_evidence(problem_profile))


,development_products,class_count,minimum_class_products,maximum_class_products,grayscale_images,unusual_geometry_images,classes_with_fold_warnings,classes_untrainable_in_any_fold
0,32773,124,1,5748,294,12,26,12


,evidence,problem,choice_to_test
0,12 development images differ from the usual 60...,stretching changes shape and centre crops can ...,aspect-preserving 60-by-80 white canvas
1,294 development images are grayscale,input channel formats are inconsistent,deterministic RGB conversion for every model f...
2,124 classes range from 1 to 5748 products,accuracy can hide failure on rare classes,fixed-class macro-F1 plus per-class evidence
3,12 classes are untrainable in at least one fold,some validation folds cannot represent every c...,fixed-class macro-F1 with zero-support classes...
4,product shape and edge detail may separate vis...,raw pixels may not provide a strong low-data b...,HOG with k-NN and linear SVM baselines
5,32773 development products support a small ben...,a large network can overfit a limited labelled...,small scratch CNN against the classical baselines
6,26 classes have rare-class fold warnings,rare classes see too few distinct training exa...,training-only augmentation for the scratch CNN


### Observation — some classes have very little data

The development set contains 32,773 products across 124 classes. Class sizes range from just 1 image to 5,748. Twenty-six classes have fold warnings, and 12 are missing from at least one fold's training data. A model can therefore score well on common products while failing on rare ones.

We used macro-F1 to give each class equal importance and tested class weighting to give rare classes more influence during training. Neither choice can create the missing examples a model needs to learn a class.

The data also contains 294 grayscale images and 12 images with unusual shapes. This led us to use a common colour format and resize with padding, preserving product shape.

## 3. Fair experiment setup

Every candidate used the five development folds saved in `data/processed/splits.csv`. A fold is one fixed group of images. Using the same groups makes comparisons fairer because each model faces the same validation products.

Only image pixels were used as inputs. Product names, years, and other target labels were excluded to keep the task focused on visual recognition. Image normalization and class weights were fitted using each fold's training rows only.

Task 1 model selection used development results, not Task 1 holdout scores. Other team members had already used the shared holdout for their targets, so it cannot be described as untouched across the whole project.

## 4. Evaluation

For each final candidate, we trained on four folds and validated on the fifth, repeating this across all five folds. Each product received an **out-of-fold (OOF) prediction** from a model that had not trained on that product.

Our main score was **macro-F1**: the average of each class's F1 score, which balances finding the right products with avoiding wrong labels. All 124 classes remain in the calculation, including classes with zero F1. The mean reports the average fold score; the standard deviation after ± shows how much the five fold scores differ.

We also measured Top-1 accuracy (the first answer is correct), Top-5 accuracy (the correct answer appears among five choices), and weighted F1 (common classes count more). Per-class scores and confusion pairs show which products fail.

**Evaluation limit:** These are development scores. Validation folds also guided CNN checkpoint selection, and fold 0 guided classical tuning. They are not an independent final test. The saved validation-loss values averaged batches equally rather than images, so they are excluded from model selection and curve analysis.

## 5. Candidate hypotheses

We organised the investigation around four questions:

| Experiment | Question |
|---|---|
| Plain scratch CNN | How well can a small model learn image features directly? |
| HOG with k-NN and SVM | Can simpler edge-and-shape features compete? |
| CNN with mild augmentation | Do small changes to training images help the model generalise? |
| Augmented CNN with class weights | Does giving rare classes more influence improve macro-F1? |

The CNN comparisons kept the architecture and training recipe fixed. This let us study augmentation first, then weighting, without changing both at once.

## 6. Controlled preprocessing

Images were oriented correctly, converted to RGB (three colour channels), and resized onto a white canvas 60 pixels wide and 80 pixels high. Padding preserved their shape instead of stretching them. RGB normalization used training-only channel averages and spreads to put input values on a common scale.

The plain CNN used no random image changes. The augmentation version added horizontal flips, rotations up to 5°, shifts up to 5%, scale changes from 0.95 to 1.05, and brightness and contrast factors from 0.9 to 1.1. These mild changes were intended to vary the view while keeping the article type recognisable. Validation images received no random augmentation.

In [2]:
from fashion.task1 import (
    DEFAULT_TASK1_PREPROCESSING,
    TASK1_CONTROL_PREPROCESSING,
    TASK1_BALANCED_WEIGHTED_CANDIDATE,
    TASK1_MILD_AUG_CANDIDATE,
    TASK1_NO_AUG_CANDIDATE,
    run_task1_classical_experiment,
    run_task1_experiment,
)

preprocessing_candidates = {
    "control_no_augmentation": TASK1_CONTROL_PREPROCESSING.to_dict(),
    "hypothesis_mild_augmentation": DEFAULT_TASK1_PREPROCESSING.to_dict(),
}
cnn_candidates = pd.DataFrame(
    [
        {
            "candidate_id": candidate.candidate_id,
            "preprocessing_id": candidate.preprocessing.preprocessing_id,
            "loss_id": candidate.loss.loss_id,
        }
        for candidate in (
            TASK1_NO_AUG_CANDIDATE,
            TASK1_MILD_AUG_CANDIDATE,
            TASK1_BALANCED_WEIGHTED_CANDIDATE,
        )
    ]
)

display(pd.DataFrame(preprocessing_candidates).T)
display(cnn_candidates)


,preprocessing_id,image_size,pad_color,horizontal_flip_probability,max_rotation_degrees,max_translation_fraction,scale_range,brightness_range,contrast_range
control_no_augmentation,task1_rgb_60x80_no_aug_v1,"(80, 60)","(255, 255, 255)",0.0,0.0,0.0,"(1.0, 1.0)","(1.0, 1.0)","(1.0, 1.0)"
hypothesis_mild_augmentation,task1_rgb_60x80_mild_aug_v1,"(80, 60)","(255, 255, 255)",0.5,5.0,0.05,"(0.95, 1.05)","(0.9, 1.1)","(0.9, 1.1)"


,candidate_id,preprocessing_id,loss_id
0,task1_cnn_no_aug_unweighted_v1,task1_rgb_60x80_no_aug_v1,cross_entropy_unweighted_v1
1,task1_cnn_mild_aug_unweighted_v1,task1_rgb_60x80_mild_aug_v1,cross_entropy_unweighted_v1
2,task1_cnn_mild_aug_balanced_weighted_v1,task1_rgb_60x80_mild_aug_v1,cross_entropy_balanced_class_weighted_v1


### Observation — each CNN comparison isolates one change

The displayed settings show the shared image preparation and the three preprocessing/loss combinations. The plain-versus-augmented comparison changes the training images. The augmented-versus-weighted comparison keeps those image changes and alters the loss instead. Architecture, folds, and training budget are fixed in the experiment code.

## 7. Scratch CNN control

**Question:** How well could a small CNN learn directly from these images?

A CNN learns visual patterns through small filters. Our model used five convolution layers with 32, 64, 64, 64, and 64 channels. Pooling reduced the image features, followed by a 128-unit layer and 124 output scores. This gave us a compact model to use as the common starting point for later tests.

Each fold trained for 20 epochs, meaning 20 passes through its training data, in batches of 128. Adam updated the weights with a one-cycle learning-rate schedule peaking at 0.001. Weight decay was 0.00001 and gradient clipping was 1.0; these limited weight growth and large updates. These were fixed recipe settings, not a claimed tuning search.

The loss was ordinary cross-entropy, which penalises wrong class predictions equally across classes. We saved the checkpoint with the highest validation macro-F1 in each fold. The following tables show the completed control results.

In [ ]:
from fashion.config import TASK1_EVIDENCE_DIR

scratch_candidate_id = "task1_cnn_no_aug_unweighted_v1"
saved_cnn_folds = pd.read_csv(TASK1_EVIDENCE_DIR / "fold_metrics.csv", keep_default_na=False)
saved_cnn_comparison = pd.read_csv(TASK1_EVIDENCE_DIR / "comparison.csv", keep_default_na=False)
saved_cnn_oof = pd.read_csv(TASK1_EVIDENCE_DIR / "oof_metrics.csv", keep_default_na=False)

display(saved_cnn_folds.loc[saved_cnn_folds["candidate_id"].eq(scratch_candidate_id)])
display(saved_cnn_comparison.loc[saved_cnn_comparison["candidate_id"].eq(scratch_candidate_id)])
display(saved_cnn_oof.loc[saved_cnn_oof["candidate_id"].eq(scratch_candidate_id)])

run_id,fold,candidate_id,preprocessing_id,loss_id,macro_f1,weighted_f1,top1_accuracy,top5_accuracy,validation_loss
task1-cnn-task1_cnn_no_aug_unweighted_v1-f0-s2753-2b671c263f75,0,task1_cnn_no_aug_unweighted_v1,task1_rgb_60x80_no_aug_v1,cross_entropy_unweighted_v1,0.498052,0.834539,0.837784,0.977873,1.072126
task1-cnn-task1_cnn_no_aug_unweighted_v1-f1-s2753-326863573f92,1,task1_cnn_no_aug_unweighted_v1,task1_rgb_60x80_no_aug_v1,cross_entropy_unweighted_v1,0.519482,0.844484,0.848231,0.978951,1.058708
task1-cnn-task1_cnn_no_aug_unweighted_v1-f2-s2753-4a2ad5b813b3,2,task1_cnn_no_aug_unweighted_v1,task1_rgb_60x80_no_aug_v1,cross_entropy_unweighted_v1,0.517600,0.826542,0.832138,0.973447,1.253813
task1-cnn-task1_cnn_no_aug_unweighted_v1-f3-s2753-cbd14501bc35,3,task1_cnn_no_aug_unweighted_v1,task1_rgb_60x80_no_aug_v1,cross_entropy_unweighted_v1,0.536994,0.839710,0.843302,0.978792,1.168490
task1-cnn-task1_cnn_no_aug_unweighted_v1-f4-s2753-15a400dd82cb,4,task1_cnn_no_aug_unweighted_v1,task1_rgb_60x80_no_aug_v1,cross_entropy_unweighted_v1,0.585240,0.845254,0.849474,0.980784,1.111062


candidate_id,preprocessing_id,loss_id,macro_f1_mean,macro_f1_std,weighted_f1_mean,weighted_f1_std,top1_accuracy_mean,top1_accuracy_std,top5_accuracy_mean,top5_accuracy_std
task1_cnn_no_aug_unweighted_v1,task1_rgb_60x80_no_aug_v1,cross_entropy_unweighted_v1,0.531473,0.03307,0.838106,0.007759,0.842186,0.007267,0.977969,0.002739


candidate_id,preprocessing_id,loss_id,macro_f1_124,weighted_f1,top1_accuracy,top5_accuracy
task1_cnn_no_aug_unweighted_v1,task1_rgb_60x80_no_aug_v1,cross_entropy_unweighted_v1,0.556926,0.839208,0.842187,0.97797


### Observation — the CNN establishes the reference score

The plain CNN reached **0.5315 ± 0.0331 macro-F1**, with 84.22% Top-1 accuracy and 97.80% Top-5 accuracy. Its fold scores ranged from 0.4981 to 0.5852, showing that performance differed across the five image groups.

The high accuracy alongside the much lower macro-F1 suggests that recognising common products is easier than doing well across all classes. We examine this in the class-error analysis below.

**Decision:** Use this model as the reference for the simpler models and the two CNN changes.

## 8. HOG baselines

**Question:** Could hand-built shape features compete with learned CNN features?

HOG describes the directions of edges in small image regions. We extracted these features from grayscale images. k-NN chose labels from similar training examples; a linear SVM learned boundaries between classes in this feature space.

Tuning used saved fold 0. We compared HOG cells of 16 × 16 and 10 × 10 pixels, then tested neighbour counts of 3, 5, and 11 with equal or distance-based votes. For SVM, we tested C values of 0.1, 1, and 10 with and without class balancing. C controls how strongly training mistakes are penalised.

Macro-F1 guided the choice of 10 × 10 HOG cells, distance-weighted 3-neighbour k-NN, and a balanced SVM with C = 0.1. The selected settings were then evaluated across all five folds. This reused fold 0, so the comparison remains development evidence.

In [8]:
CLASSICAL_STAGE = "smoke"  # Use "tune" and then "final" only as separate deliberate runs.
classic_experiment = run_task1_classical_experiment(
    splits,
    article_type_map,
    stage=CLASSICAL_STAGE,
)
display(classic_experiment.fold_metrics)
if not classic_experiment.tuning.empty:
    display(classic_experiment.tuning)
if not classic_experiment.comparison.empty:
    display(classic_experiment.comparison)
if not classic_experiment.oof_metrics.empty:
    display(classic_experiment.oof_metrics)


,run_id,fold,candidate_id,hog_id,model_family,macro_f1,weighted_f1,top1_accuracy,top5_accuracy
0,task1-classical-task1_gray_hog_ppc10_v1-knn-k3...,0,task1_gray_hog_ppc10_v1-knn-k3-distance,task1_gray_hog_ppc10_v1,task1_hog_knn_v1,0.489685,0.789193,0.798413,0.890279
1,task1-classical-task1_gray_hog_ppc10_v1-knn-k3...,1,task1_gray_hog_ppc10_v1-knn-k3-distance,task1_gray_hog_ppc10_v1,task1_hog_knn_v1,0.496920,0.782313,0.791489,0.898414
2,task1-classical-task1_gray_hog_ppc10_v1-knn-k3...,2,task1_gray_hog_ppc10_v1-knn-k3-distance,task1_gray_hog_ppc10_v1,task1_hog_knn_v1,0.496270,0.780333,0.790325,0.891958
3,task1-classical-task1_gray_hog_ppc10_v1-knn-k3...,3,task1_gray_hog_ppc10_v1-knn-k3-distance,task1_gray_hog_ppc10_v1,task1_hog_knn_v1,0.502577,0.786799,0.795545,0.899298
4,task1-classical-task1_gray_hog_ppc10_v1-knn-k3...,4,task1_gray_hog_ppc10_v1-knn-k3-distance,task1_gray_hog_ppc10_v1,task1_hog_knn_v1,0.525995,0.791549,0.801891,0.896142
5,task1-classical-task1_gray_hog_ppc10_v1-linear...,0,task1_gray_hog_ppc10_v1-linear-svm-c0.1-balanced,task1_gray_hog_ppc10_v1,task1_hog_linear_svm_v1,0.462375,0.782172,0.778117,0.946284
6,task1-classical-task1_gray_hog_ppc10_v1-linear...,1,task1_gray_hog_ppc10_v1-linear-svm-c0.1-balanced,task1_gray_hog_ppc10_v1,task1_hog_linear_svm_v1,0.502251,0.784686,0.779134,0.943868
7,task1-classical-task1_gray_hog_ppc10_v1-linear...,2,task1_gray_hog_ppc10_v1-linear-svm-c0.1-balanced,task1_gray_hog_ppc10_v1,task1_hog_linear_svm_v1,0.500543,0.772063,0.769113,0.938501
8,task1-classical-task1_gray_hog_ppc10_v1-linear...,3,task1_gray_hog_ppc10_v1-linear-svm-c0.1-balanced,task1_gray_hog_ppc10_v1,task1_hog_linear_svm_v1,0.483450,0.785299,0.781355,0.948886
9,task1-classical-task1_gray_hog_ppc10_v1-linear...,4,task1_gray_hog_ppc10_v1-linear-svm-c0.1-balanced,task1_gray_hog_ppc10_v1,task1_hog_linear_svm_v1,0.495666,0.777008,0.774592,0.945402


,candidate_id,hog_id,model_family,macro_f1_mean,macro_f1_std,weighted_f1_mean,weighted_f1_std,top1_accuracy_mean,top1_accuracy_std,top5_accuracy_mean,top5_accuracy_std
0,task1_gray_hog_ppc10_v1-knn-k3-distance,task1_gray_hog_ppc10_v1,task1_hog_knn_v1,0.502289,0.014018,0.786037,0.004673,0.795533,0.004801,0.895218,0.003960
1,task1_gray_hog_ppc10_v1-linear-svm-c0.1-balanced,task1_gray_hog_ppc10_v1,task1_hog_linear_svm_v1,0.488857,0.016529,0.780246,0.005622,0.776462,0.004778,0.944588,0.003859


,candidate_id,macro_f1,weighted_f1,top1_accuracy,top5_accuracy
0,task1_gray_hog_ppc10_v1-knn-k3-distance,0.526879,0.787157,0.795533,0.895219
1,task1_gray_hog_ppc10_v1-linear-svm-c0.1-balanced,0.505912,0.780849,0.776462,0.944589


### Observation — k-NN is the stronger classical baseline

HOG k-NN reached **0.5023 ± 0.0140 macro-F1** and 79.55% Top-1 accuracy. HOG SVM reached **0.4889 ± 0.0165 macro-F1** and 77.65% Top-1 accuracy. Both scored below the plain CNN on the main metric.

SVM had better Top-5 accuracy: 94.46% versus k-NN's 89.52%. It more often included the correct label in a short list, but k-NN was better at choosing one label.

**Decision:** Keep k-NN as the strongest classical baseline. The CNN remains the leading candidate for single-label prediction.

## 9. Mild data augmentation

**Question:** Would small changes to the training images help the CNN recognise unseen products?

We added the mild image changes defined above to the plain CNN. The idea was to reduce dependence on exact poses, positions, and lighting. Architecture, loss, folds, image size, and training budget stayed the same. The tables compare the plain and augmented versions.

In [9]:
RUN_MODE = "smoke"  # Full runs are deliberate separate runs for the ten registered unweighted CNN folds.
task1_experiment = run_task1_experiment(
    splits,
    article_type_map,
    mode=RUN_MODE,
)
display(task1_experiment.fold_metrics)
if not task1_experiment.comparison.empty:
    display(task1_experiment.comparison)
if not task1_experiment.oof_metrics.empty:
    display(task1_experiment.oof_metrics)


,run_id,fold,candidate_id,preprocessing_id,loss_id,macro_f1,weighted_f1,top1_accuracy,top5_accuracy,validation_loss
0,task1-cnn-task1_cnn_no_aug_unweighted_v1-f0-s2...,0,task1_cnn_no_aug_unweighted_v1,task1_rgb_60x80_no_aug_v1,cross_entropy_unweighted_v1,0.498052,0.834539,0.837784,0.977873,1.072126
1,task1-cnn-task1_cnn_no_aug_unweighted_v1-f1-s2...,1,task1_cnn_no_aug_unweighted_v1,task1_rgb_60x80_no_aug_v1,cross_entropy_unweighted_v1,0.519482,0.844484,0.848231,0.978951,1.058708
2,task1-cnn-task1_cnn_no_aug_unweighted_v1-f2-s2...,2,task1_cnn_no_aug_unweighted_v1,task1_rgb_60x80_no_aug_v1,cross_entropy_unweighted_v1,0.517600,0.826542,0.832138,0.973447,1.253813
3,task1-cnn-task1_cnn_no_aug_unweighted_v1-f3-s2...,3,task1_cnn_no_aug_unweighted_v1,task1_rgb_60x80_no_aug_v1,cross_entropy_unweighted_v1,0.536994,0.839710,0.843302,0.978792,1.168490
4,task1-cnn-task1_cnn_no_aug_unweighted_v1-f4-s2...,4,task1_cnn_no_aug_unweighted_v1,task1_rgb_60x80_no_aug_v1,cross_entropy_unweighted_v1,0.585240,0.845254,0.849474,0.980784,1.111062
5,task1-cnn-task1_cnn_mild_aug_unweighted_v1-f0-...,0,task1_cnn_mild_aug_unweighted_v1,task1_rgb_60x80_mild_aug_v1,cross_entropy_unweighted_v1,0.508068,0.834493,0.840836,0.979551,0.581812
6,task1-cnn-task1_cnn_mild_aug_unweighted_v1-f1-...,1,task1_cnn_mild_aug_unweighted_v1,task1_rgb_60x80_mild_aug_v1,cross_entropy_unweighted_v1,0.521633,0.843648,0.848688,0.984137,0.554290
7,task1-cnn-task1_cnn_mild_aug_unweighted_v1-f2-...,2,task1_cnn_mild_aug_unweighted_v1,task1_rgb_60x80_mild_aug_v1,cross_entropy_unweighted_v1,0.523251,0.829862,0.836563,0.978483,0.679289
8,task1-cnn-task1_cnn_mild_aug_unweighted_v1-f3-...,3,task1_cnn_mild_aug_unweighted_v1,task1_rgb_60x80_mild_aug_v1,cross_entropy_unweighted_v1,0.508891,0.834636,0.837962,0.980623,0.598044
9,task1-cnn-task1_cnn_mild_aug_unweighted_v1-f4-...,4,task1_cnn_mild_aug_unweighted_v1,task1_rgb_60x80_mild_aug_v1,cross_entropy_unweighted_v1,0.547079,0.844106,0.850541,0.981851,0.604823


,candidate_id,preprocessing_id,loss_id,macro_f1_mean,macro_f1_std,weighted_f1_mean,weighted_f1_std,top1_accuracy_mean,top1_accuracy_std,top5_accuracy_mean,top5_accuracy_std
0,task1_cnn_no_aug_unweighted_v1,task1_rgb_60x80_no_aug_v1,cross_entropy_unweighted_v1,0.531473,0.033070,0.838106,0.007759,0.842186,0.007267,0.977969,0.002739
1,task1_cnn_mild_aug_unweighted_v1,task1_rgb_60x80_mild_aug_v1,cross_entropy_unweighted_v1,0.521784,0.015783,0.837349,0.006263,0.842918,0.006338,0.980929,0.002186


,candidate_id,preprocessing_id,loss_id,macro_f1_124,weighted_f1,top1_accuracy,top5_accuracy
0,task1_cnn_no_aug_unweighted_v1,task1_rgb_60x80_no_aug_v1,cross_entropy_unweighted_v1,0.556926,0.839208,0.842187,0.977970
1,task1_cnn_mild_aug_unweighted_v1,task1_rgb_60x80_mild_aug_v1,cross_entropy_unweighted_v1,0.540755,0.838582,0.842919,0.980929


### Observation — augmentation lowers fold variation and mean macro-F1

Augmentation reached **0.5218 ± 0.0158 macro-F1**, below the plain CNN by 0.0097. Top-1 accuracy was almost unchanged, at 84.29% versus 84.22%. Top-5 accuracy rose slightly, from 97.80% to 98.09%.

The smaller standard deviation means the augmented model's scores were closer across these five folds. It does not establish stability across training seeds or robustness to customer photos.

**Decision:** The hypothesis of improved macro-F1 did not pass. Keep augmentation as a useful trade-off: less fold variation, but a lower mean score.

## 10. Balanced class-weighted loss

**Question:** Could stronger penalties for mistakes on rare classes improve macro-F1?

We kept the augmented CNN and changed only the training loss. Each class present in a training fold received `weight = training rows / (present classes × class rows)`. This gave rare classes more influence than common ones. Weights used training rows only; absent classes received zero weight.

The intended benefit was better rare-class recognition. The risk was giving very large weights to classes with too few distinct examples, harming other predictions. We checked overall scores and the number of classes with zero F1 to judge that trade-off.

In [11]:
from fashion.task1 import run_task1_weighted_experiment

WEIGHTED_MODE = "smoke"  # Full runs are deliberate separate runs for the five new weighted folds.
weighted_experiment = run_task1_weighted_experiment(
    splits,
    article_type_map,
    mode=WEIGHTED_MODE,
)
display(weighted_experiment.fold_metrics)
if not weighted_experiment.comparison.empty:
    display(weighted_experiment.comparison)
    display(weighted_experiment.oof_metrics)


,run_id,fold,candidate_id,preprocessing_id,loss_id,macro_f1,weighted_f1,top1_accuracy,top5_accuracy,validation_loss
0,task1-cnn-task1_cnn_no_aug_unweighted_v1-f0-s2...,0,task1_cnn_no_aug_unweighted_v1,task1_rgb_60x80_no_aug_v1,cross_entropy_unweighted_v1,0.498052,0.834539,0.837784,0.977873,1.072126
1,task1-cnn-task1_cnn_no_aug_unweighted_v1-f1-s2...,1,task1_cnn_no_aug_unweighted_v1,task1_rgb_60x80_no_aug_v1,cross_entropy_unweighted_v1,0.519482,0.844484,0.848231,0.978951,1.058708
2,task1-cnn-task1_cnn_no_aug_unweighted_v1-f2-s2...,2,task1_cnn_no_aug_unweighted_v1,task1_rgb_60x80_no_aug_v1,cross_entropy_unweighted_v1,0.517600,0.826542,0.832138,0.973447,1.253813
3,task1-cnn-task1_cnn_no_aug_unweighted_v1-f3-s2...,3,task1_cnn_no_aug_unweighted_v1,task1_rgb_60x80_no_aug_v1,cross_entropy_unweighted_v1,0.536994,0.839710,0.843302,0.978792,1.168490
4,task1-cnn-task1_cnn_no_aug_unweighted_v1-f4-s2...,4,task1_cnn_no_aug_unweighted_v1,task1_rgb_60x80_no_aug_v1,cross_entropy_unweighted_v1,0.585240,0.845254,0.849474,0.980784,1.111062
5,task1-cnn-task1_cnn_mild_aug_unweighted_v1-f0-...,0,task1_cnn_mild_aug_unweighted_v1,task1_rgb_60x80_mild_aug_v1,cross_entropy_unweighted_v1,0.508068,0.834493,0.840836,0.979551,0.581812
6,task1-cnn-task1_cnn_mild_aug_unweighted_v1-f1-...,1,task1_cnn_mild_aug_unweighted_v1,task1_rgb_60x80_mild_aug_v1,cross_entropy_unweighted_v1,0.521633,0.843648,0.848688,0.984137,0.554290
7,task1-cnn-task1_cnn_mild_aug_unweighted_v1-f2-...,2,task1_cnn_mild_aug_unweighted_v1,task1_rgb_60x80_mild_aug_v1,cross_entropy_unweighted_v1,0.523251,0.829862,0.836563,0.978483,0.679289
8,task1-cnn-task1_cnn_mild_aug_unweighted_v1-f3-...,3,task1_cnn_mild_aug_unweighted_v1,task1_rgb_60x80_mild_aug_v1,cross_entropy_unweighted_v1,0.508891,0.834636,0.837962,0.980623,0.598044
9,task1-cnn-task1_cnn_mild_aug_unweighted_v1-f4-...,4,task1_cnn_mild_aug_unweighted_v1,task1_rgb_60x80_mild_aug_v1,cross_entropy_unweighted_v1,0.547079,0.844106,0.850541,0.981851,0.604823


,candidate_id,preprocessing_id,loss_id,macro_f1_mean,macro_f1_std,weighted_f1_mean,weighted_f1_std,top1_accuracy_mean,top1_accuracy_std,top5_accuracy_mean,top5_accuracy_std
0,task1_cnn_no_aug_unweighted_v1,task1_rgb_60x80_no_aug_v1,cross_entropy_unweighted_v1,0.531473,0.033070,0.838106,0.007759,0.842186,0.007267,0.977969,0.002739
1,task1_cnn_mild_aug_unweighted_v1,task1_rgb_60x80_mild_aug_v1,cross_entropy_unweighted_v1,0.521784,0.015783,0.837349,0.006263,0.842918,0.006338,0.980929,0.002186
2,task1_cnn_mild_aug_balanced_weighted_v1,task1_rgb_60x80_mild_aug_v1,cross_entropy_balanced_class_weighted_v1,0.456413,0.018133,0.713878,0.024425,0.691510,0.024893,0.952491,0.007479


,candidate_id,preprocessing_id,loss_id,macro_f1_124,weighted_f1,top1_accuracy,top5_accuracy
0,task1_cnn_no_aug_unweighted_v1,task1_rgb_60x80_no_aug_v1,cross_entropy_unweighted_v1,0.556926,0.839208,0.842187,0.977970
1,task1_cnn_mild_aug_unweighted_v1,task1_rgb_60x80_mild_aug_v1,cross_entropy_unweighted_v1,0.540755,0.838582,0.842919,0.980929
2,task1_cnn_mild_aug_balanced_weighted_v1,task1_rgb_60x80_mild_aug_v1,cross_entropy_balanced_class_weighted_v1,0.466383,0.714510,0.691514,0.952491


### Observation — full class balancing hurts overall performance

The weighted CNN reached **0.4564 ± 0.0181 macro-F1**, 69.15% Top-1 accuracy, and 0.7139 weighted F1. All three were below the augmented unweighted CNN.

The number of classes with zero pooled OOF F1 fell only from 21 to 20. This small change did not offset the wider loss in performance. One possible explanation is that weighting increased the influence of scarce examples without adding the variety needed to learn those classes. This experiment does not prove that cause.

**Decision:** Reject this full inverse-frequency weighting recipe. It does not show that all forms of class weighting would fail.

## 11. Combined five-fold and OOF comparison

We now bring the experiment results together. The tables below contain the three CNN candidates; the classical results appear in Section 8.

The fold mean averages five separate macro-F1 scores. **Pooled OOF macro-F1** instead scores all out-of-fold predictions together. F1 is calculated from combined prediction counts, so these two summaries need not match. We use the declared fold-mean metric for selection and the pooled results as supporting evidence.

In [13]:
from fashion.config import TASK1_EVIDENCE_DIR

for evidence_name in ("fold_metrics", "comparison", "oof_metrics"):
    evidence_path = TASK1_EVIDENCE_DIR / f"{evidence_name}.csv"
    if evidence_path.is_file():
        print(evidence_name)
        display(pd.read_csv(evidence_path, keep_default_na=False))
    else:
        print(f"{evidence_name} is not ready; complete the needed full runs first.")


fold_metrics


,run_id,fold,candidate_id,preprocessing_id,loss_id,macro_f1,weighted_f1,top1_accuracy,top5_accuracy,validation_loss
0,task1-cnn-task1_cnn_no_aug_unweighted_v1-f0-s2...,0,task1_cnn_no_aug_unweighted_v1,task1_rgb_60x80_no_aug_v1,cross_entropy_unweighted_v1,0.498052,0.834539,0.837784,0.977873,1.072126
1,task1-cnn-task1_cnn_no_aug_unweighted_v1-f1-s2...,1,task1_cnn_no_aug_unweighted_v1,task1_rgb_60x80_no_aug_v1,cross_entropy_unweighted_v1,0.519482,0.844484,0.848231,0.978951,1.058708
2,task1-cnn-task1_cnn_no_aug_unweighted_v1-f2-s2...,2,task1_cnn_no_aug_unweighted_v1,task1_rgb_60x80_no_aug_v1,cross_entropy_unweighted_v1,0.517600,0.826542,0.832138,0.973447,1.253813
3,task1-cnn-task1_cnn_no_aug_unweighted_v1-f3-s2...,3,task1_cnn_no_aug_unweighted_v1,task1_rgb_60x80_no_aug_v1,cross_entropy_unweighted_v1,0.536994,0.839710,0.843302,0.978792,1.168490
4,task1-cnn-task1_cnn_no_aug_unweighted_v1-f4-s2...,4,task1_cnn_no_aug_unweighted_v1,task1_rgb_60x80_no_aug_v1,cross_entropy_unweighted_v1,0.585240,0.845254,0.849474,0.980784,1.111062
5,task1-cnn-task1_cnn_mild_aug_unweighted_v1-f0-...,0,task1_cnn_mild_aug_unweighted_v1,task1_rgb_60x80_mild_aug_v1,cross_entropy_unweighted_v1,0.508068,0.834493,0.840836,0.979551,0.581812
6,task1-cnn-task1_cnn_mild_aug_unweighted_v1-f1-...,1,task1_cnn_mild_aug_unweighted_v1,task1_rgb_60x80_mild_aug_v1,cross_entropy_unweighted_v1,0.521633,0.843648,0.848688,0.984137,0.554290
7,task1-cnn-task1_cnn_mild_aug_unweighted_v1-f2-...,2,task1_cnn_mild_aug_unweighted_v1,task1_rgb_60x80_mild_aug_v1,cross_entropy_unweighted_v1,0.523251,0.829862,0.836563,0.978483,0.679289
8,task1-cnn-task1_cnn_mild_aug_unweighted_v1-f3-...,3,task1_cnn_mild_aug_unweighted_v1,task1_rgb_60x80_mild_aug_v1,cross_entropy_unweighted_v1,0.508891,0.834636,0.837962,0.980623,0.598044
9,task1-cnn-task1_cnn_mild_aug_unweighted_v1-f4-...,4,task1_cnn_mild_aug_unweighted_v1,task1_rgb_60x80_mild_aug_v1,cross_entropy_unweighted_v1,0.547079,0.844106,0.850541,0.981851,0.604823


comparison


,candidate_id,preprocessing_id,loss_id,macro_f1_mean,macro_f1_std,weighted_f1_mean,weighted_f1_std,top1_accuracy_mean,top1_accuracy_std,top5_accuracy_mean,top5_accuracy_std
0,task1_cnn_no_aug_unweighted_v1,task1_rgb_60x80_no_aug_v1,cross_entropy_unweighted_v1,0.531473,0.033070,0.838106,0.007759,0.842186,0.007267,0.977969,0.002739
1,task1_cnn_mild_aug_unweighted_v1,task1_rgb_60x80_mild_aug_v1,cross_entropy_unweighted_v1,0.521784,0.015783,0.837349,0.006263,0.842918,0.006338,0.980929,0.002186
2,task1_cnn_mild_aug_balanced_weighted_v1,task1_rgb_60x80_mild_aug_v1,cross_entropy_balanced_class_weighted_v1,0.456413,0.018133,0.713878,0.024425,0.691510,0.024893,0.952491,0.007479


oof_metrics


,candidate_id,preprocessing_id,loss_id,macro_f1_124,weighted_f1,top1_accuracy,top5_accuracy
0,task1_cnn_no_aug_unweighted_v1,task1_rgb_60x80_no_aug_v1,cross_entropy_unweighted_v1,0.556926,0.839208,0.842187,0.977970
1,task1_cnn_mild_aug_unweighted_v1,task1_rgb_60x80_mild_aug_v1,cross_entropy_unweighted_v1,0.540755,0.838582,0.842919,0.980929
2,task1_cnn_mild_aug_balanced_weighted_v1,task1_rgb_60x80_mild_aug_v1,cross_entropy_balanced_class_weighted_v1,0.466383,0.714510,0.691514,0.952491


In [14]:
from fashion.config import TASK1_FIGURE_DIR
from fashion.task1 import write_task1_comparison_figure, write_task1_confusion_figure

if WEIGHTED_MODE == "full" and not weighted_experiment.fold_metrics.empty:
    write_task1_comparison_figure(weighted_experiment.fold_metrics)
    for candidate_id, predictions in weighted_experiment.oof_predictions.items():
        write_task1_confusion_figure(
            predictions,
            article_type_classes,
            output=TASK1_FIGURE_DIR / f"cnn_oof_confusion_{candidate_id}.png",
        )
elif RUN_MODE == "full" and not task1_experiment.fold_metrics.empty:
    write_task1_comparison_figure(task1_experiment.fold_metrics)
    for candidate_id, predictions in task1_experiment.oof_predictions.items():
        write_task1_confusion_figure(
            predictions,
            article_type_classes,
            output=TASK1_FIGURE_DIR / f"cnn_oof_confusion_{candidate_id}.png",
        )

if CLASSICAL_STAGE == "final":
    for candidate_id, predictions in classic_experiment.oof_predictions.items():
        write_task1_confusion_figure(
            predictions,
            article_type_classes,
            output=TASK1_FIGURE_DIR / f"classical_oof_confusion_{candidate_id}.png",
        )


### Observation — the plain CNN leads on the chosen metric

| Candidate | Mean macro-F1 ± fold standard deviation |
|---|---|
| Plain CNN | **0.5315 ± 0.0331** |
| Augmented CNN | 0.5218 ± 0.0158 |
| HOG k-NN | 0.5023 ± 0.0140 |
| HOG SVM | 0.4889 ± 0.0165 |
| Weighted augmented CNN | 0.4564 ± 0.0181 |

Source: saved CNN `comparison.csv` and `classical_comparison.csv` in `results/evidence/task1/`.

The pooled CNN scores support the same ranking: 0.5569 for the plain model, 0.5408 with augmentation, and 0.4664 with weighting. The plain CNN leads on both summaries, although augmentation has lower fold variation.

**Decision:** Select the plain CNN on macro-F1. Its 0.0097 lead over augmentation is an observed development difference, not proof of a reliable gain across random seeds. We still need to understand its learning behaviour and failures.

## 12. Learning-curve diagnosis

**Question:** Did the CNN keep learning useful patterns, or mainly become better at its training images?

The plots average the five training histories across 20 epochs. The left chart shows training loss; the right shows validation macro-F1. Shaded bands show variation between folds. Reading the two together helps identify overfitting: training improves while performance on unseen fold images stops improving.

In [12]:
from collections import defaultdict
from fashion.config import ROOT, TASK1_EVIDENCE_DIR, TASK1_FIGURE_DIR
from fashion.task1 import write_task1_learning_curve_figure

expected_learning_candidates = {
    TASK1_NO_AUG_CANDIDATE.candidate_id,
    TASK1_MILD_AUG_CANDIDATE.candidate_id,
    TASK1_BALANCED_WEIGHTED_CANDIDATE.candidate_id,
}
fold_metrics_path = TASK1_EVIDENCE_DIR / "fold_metrics.csv"
histories_path = TASK1_EVIDENCE_DIR / "cnn_learning_histories.csv"
learning_curve_histories = defaultdict(list)

if not fold_metrics_path.is_file() or not histories_path.is_file():
    print("Learning curves are not ready; compact Task 1 evidence is missing.")
else:
    combined_folds = pd.read_csv(fold_metrics_path, keep_default_na=False)
    compact_histories = pd.read_csv(histories_path, keep_default_na=False)
    required_columns = {"run_id", "candidate_id", "fold"}
    required_history_columns = required_columns | {"epoch", "train_loss", "macro_f1"}
    has_required_columns = required_columns.issubset(combined_folds.columns)
    has_history_columns = required_history_columns.issubset(compact_histories.columns)
    has_complete_folds = False
    if has_required_columns and has_history_columns:
        fold_counts = combined_folds.groupby("candidate_id")["fold"].nunique()
        has_complete_folds = (
            len(combined_folds) == 15
            and set(combined_folds["candidate_id"]) == expected_learning_candidates
            and set(fold_counts) == {5}
            and set(compact_histories["run_id"]) == set(combined_folds["run_id"])
        )

    if not has_complete_folds:
        print("Learning curves are not ready; compact Task 1 evidence is incomplete.")
    else:
        for (candidate_id, _), history in compact_histories.groupby(
            ["candidate_id", "fold"], sort=True
        ):
            learning_curve_histories[str(candidate_id)].append(
                history.sort_values("epoch", kind="stable")
            )

        if any(len(items) != 5 for items in learning_curve_histories.values()):
            print("Learning curves are not ready; compact Task 1 evidence is incomplete.")
        else:
            learning_curve_path = write_task1_learning_curve_figure(
                dict(learning_curve_histories),
                output=TASK1_FIGURE_DIR / "cnn_learning_curves.png",
                include_validation_loss=False,
            )
            print(f"Wrote {learning_curve_path}")




Wrote C:\Users\Khoa\Documents\MLA2\results\figures\task1\cnn_learning_curves.png


### Observation — the plain CNN shows signs of overfitting

![Mean CNN learning curves across five folds](../results/figures/task1/cnn_learning_curves.png)

The plain CNN's training loss fell close to zero, while validation macro-F1 levelled off near 0.53. This pattern suggests that later training improved its fit to known images without a matching gain on unseen ones.

The augmented CNN ended near 0.52 macro-F1 and retained a higher training loss. That is consistent with a harder, more varied training task, but it did not produce a higher validation score. The weighted CNN ended lower, near 0.46. Its loss uses class weights, so its absolute training-loss value is not directly comparable with the two unweighted models.

**Lesson:** Augmentation changed the learning behaviour, but the plain CNN still gave the best development score. Its remaining overfitting is a limit of the chosen model.

## 13. Weak-class/confusion analysis

**Question:** Which products does the model fail to recognise?

The weak-class table shows labels with low F1 and their support, meaning the number of true examples. Confusion pairs show which wrong labels the model chooses. We use these to separate rare-class failures from mistakes between similar products. Repeated candidate rows in the displayed tables refer to the same saved evidence, not additional experiments.

In [15]:
from fashion.task1 import (
    build_task1_confusion_pairs,
    build_task1_weak_class_table,
)

per_class_evidence = {
    **{f"cnn:{key}": value for key, value in task1_experiment.per_class.items()},
    **{f"weighted:{key}": value for key, value in weighted_experiment.per_class.items()},
    **{f"classic:{key}": value for key, value in classic_experiment.per_class.items()},
}
oof_prediction_evidence = {
    **{f"cnn:{key}": value for key, value in task1_experiment.oof_predictions.items()},
    **{f"weighted:{key}": value for key, value in weighted_experiment.oof_predictions.items()},
    **{f"classic:{key}": value for key, value in classic_experiment.oof_predictions.items()},
}

if per_class_evidence:
    display(build_task1_weak_class_table(per_class_evidence, limit=10))
else:
    print("Weak-class evidence is not ready; complete full/final runs first.")

if oof_prediction_evidence:
    display(build_task1_confusion_pairs(oof_prediction_evidence, limit=10))
else:
    print("Confusion-pair evidence is not ready; complete full/final runs first.")


,candidate_id,class_index,class_name,support,precision,recall,f1
0,cnn:task1_cnn_no_aug_unweighted_v1,7,Body Wash and Scrub,1,0.0,0.0,0.0
1,cnn:task1_cnn_no_aug_unweighted_v1,23,Cushion Covers,1,0.0,0.0,0.0
2,cnn:task1_cnn_no_aug_unweighted_v1,44,Ipad,1,0.0,0.0,0.0
3,cnn:task1_cnn_no_aug_unweighted_v1,51,Key chain,1,0.0,0.0,0.0
4,cnn:task1_cnn_no_aug_unweighted_v1,64,Lounge Tshirts,1,0.0,0.0,0.0
...,...,...,...,...,...,...,...
65,classic:task1_gray_hog_ppc10_v1-linear-svm-c0....,76,Rain Jacket,1,0.0,0.0,0.0
66,classic:task1_gray_hog_ppc10_v1-linear-svm-c0....,77,Rain Trousers,1,0.0,0.0,0.0
67,classic:task1_gray_hog_ppc10_v1-linear-svm-c0....,90,Shoe Laces,1,0.0,0.0,0.0
68,classic:task1_gray_hog_ppc10_v1-linear-svm-c0....,106,Ties and Cufflinks,1,0.0,0.0,0.0


,candidate_id,true_label,predicted_label,error_count,example_ids
0,cnn:task1_cnn_no_aug_unweighted_v1,Sports Shoes,Casual Shoes,286,"1548,1549,1653"
1,cnn:task1_cnn_no_aug_unweighted_v1,Casual Shoes,Sports Shoes,281,"1543,1544,1545"
2,cnn:task1_cnn_no_aug_unweighted_v1,Tshirts,Tops,246,"1570,1763,1984"
3,cnn:task1_cnn_no_aug_unweighted_v1,Tops,Tshirts,235,"2120,2294,2700"
4,cnn:task1_cnn_no_aug_unweighted_v1,Flats,Heels,141,"2609,2613,2614"
...,...,...,...,...,...
65,classic:task1_gray_hog_ppc10_v1-linear-svm-c0....,Casual Shoes,Formal Shoes,166,"1917,2368,2371"
66,classic:task1_gray_hog_ppc10_v1-linear-svm-c0....,Flats,Heels,140,"2625,2626,2628"
67,classic:task1_gray_hog_ppc10_v1-linear-svm-c0....,Heels,Flats,128,"2878,2885,2888"
68,classic:task1_gray_hog_ppc10_v1-linear-svm-c0....,Tshirts,Innerwear Vests,104,"1966,2002,2011"


### Observation — some classes are still missed entirely

![OOF confusion matrix for the selected plain unweighted CNN](../results/figures/task1/cnn_oof_confusion_task1_cnn_no_aug_unweighted_v1.png)

The selected plain CNN has **21 of 124 classes with zero pooled OOF F1**. Its overall accuracy therefore does not mean it works well for every article type.

In the matrix, the diagonal shows correct predictions and cells away from it show mistakes. The full view covers all classes, but the largest confusions are easier to read in the focused figures below.

In [ ]:
from fashion.config import ROOT, TASK1_EVIDENCE_DIR, TASK1_FIGURE_DIR
from fashion.task1 import (
    build_task1_confusion_detail,
    validate_oof_predictions,
    write_task1_confusion_example_figure,
    write_task1_confusion_pair_figure,
    write_task1_focused_confusion_figure,
)
from fashion.train.artifacts import atomic_write_csv

selected_candidate_id = "task1_cnn_no_aug_unweighted_v1"
selected_oof_predictions = pd.read_csv(
    TASK1_EVIDENCE_DIR / "selected_oof_predictions.csv",
    keep_default_na=False,
)
validate_oof_predictions(
    selected_oof_predictions, task1_development["id"].astype(int).tolist()
)
confusion_detail = build_task1_confusion_detail(
    selected_oof_predictions,
    candidate_id=selected_candidate_id,
    limit=10,
)
atomic_write_csv(TASK1_EVIDENCE_DIR / "top_confusion_pairs.csv", confusion_detail)
write_task1_confusion_pair_figure(confusion_detail)
write_task1_focused_confusion_figure(selected_oof_predictions, confusion_detail)
write_task1_confusion_example_figure(
    selected_oof_predictions,
    splits,
    confusion_detail,
)
display(confusion_detail)

,rank,candidate_id,true_label,predicted_label,error_count,true_support,error_rate,example_ids
0,1,task1_cnn_no_aug_unweighted_v1,Sports Shoes,Casual Shoes,286,1691,0.169131,"1548,1549,1653"
1,2,task1_cnn_no_aug_unweighted_v1,Casual Shoes,Sports Shoes,281,2276,0.123462,"1543,1544,1545"
2,3,task1_cnn_no_aug_unweighted_v1,Tshirts,Tops,246,5748,0.042797,"1570,1763,1984"
3,4,task1_cnn_no_aug_unweighted_v1,Tops,Tshirts,235,1369,0.171658,"2120,2294,2700"
4,5,task1_cnn_no_aug_unweighted_v1,Flats,Heels,141,356,0.396067,"2609,2613,2614"
5,6,task1_cnn_no_aug_unweighted_v1,Shirts,Tshirts,112,2619,0.042764,"2050,2095,2100"
6,7,task1_cnn_no_aug_unweighted_v1,Heels,Flats,104,890,0.116854,"2611,2624,2870"
7,8,task1_cnn_no_aug_unweighted_v1,Casual Shoes,Formal Shoes,72,2276,0.031634,"2368,2371,2376"
8,9,task1_cnn_no_aug_unweighted_v1,Tshirts,Shirts,68,5748,0.011830,"1803,1809,2155"
9,10,task1_cnn_no_aug_unweighted_v1,Shirts,Tops,64,2619,0.024437,"2119,2121,5111"


### A closer look at the largest mistakes

![Top ten directed confusion pairs](../results/figures/task1/top_confusion_pairs.png)

Frequent errors include Sports Shoes → Casual Shoes (286), Casual Shoes → Sports Shoes (281), Tshirts → Tops (246), and Tops → Tshirts (235). Counts highlight the total number of affected products, but common classes have more chances to make errors.

![Focused confusion matrix for the largest pairs](../results/figures/task1/focused_confusion_matrix.png)

The focused matrix adds the share of each true class affected. Sports Shoes → Casual Shoes accounts for 16.9% of Sports Shoes. Flats → Heels has fewer errors (141), but affects 39.6% of Flats. It is therefore a more severe failure within that class. The `Other` column includes predictions outside the displayed classes so the row totals remain complete.

![Representative images from the largest confusion pairs](../results/figures/task1/confusion_examples.png)

The examples suggest why these mistakes are plausible: shoe categories can share similar profiles, and `Tshirts` and `Tops` can overlap in shape and sleeve length. They illustrate possible visual and label-boundary difficulties; they do not prove the cause of every error.

**Lesson:** Both error counts and within-class rates matter. We recommend human review for rare or visually overlapping article types; these development results do not establish that such a review process has been tested.

## 14. Development decision and final-workflow handoff

**We selected the plain unweighted CNN, without augmentation.** It achieved the highest mean macro-F1, **0.5315 ± 0.0331**, and the highest pooled CNN macro-F1, **0.5569**. This was a primary-metric decision: each article type counts equally, which matters when class sizes vary so widely.

Augmentation did not pass the macro-F1 improvement test, although it reduced fold variation and slightly improved Top-5 accuracy. Full class weighting lowered overall performance with little benefit to zero-F1 classes. The HOG models remained useful baselines but did not beat the plain CNN.

The selected model still has 21 zero-F1 classes, confusion between similar products, and signs of overfitting. Its small lead over augmentation should be read with those limits, not treated as proof that it will always perform better.

Selected candidate: `task1_cnn_no_aug_unweighted_v1`.

The final model has since been trained from scratch on all 32,773 development images for a fixed 20 epochs, retaining the last epoch rather than a best-fold checkpoint. The saved model manifest and completed run record confirm this step. Final training, holdout evaluation, and assignment export are recorded in [Task 1 — Final Training, Evaluation and Judgement](03_task1_part2_final_evaluation.ipynb). That later evaluation assesses the chosen model; its scores are not the basis for the development choice above.